<a href="https://colab.research.google.com/github/Karol315/Hackathon_Task_1/blob/main/nn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import LogisticRegression
from skfp.fingerprints import MACCSFingerprint
from sklearn.metrics import f1_score

In [16]:
#!pip install scikit-fingerprints

In [17]:

# ==========================================
# FUNKCJA: Parsowanie hierarchii z pliku .obo
# ==========================================
def parse_obo_hierarchy(file_path):
    """
    Czyta plik .obo i zwraca słownik relacji: { 'Rodzic': ['Dziecko1', 'Dziecko2', ...] }
    """
    parent_child_map = {}
    with open(file_path, 'r', encoding='utf-8') as f:
        current_term = None
        for line in f:
            line = line.strip()
            if line.startswith('[Term]'):
                current_term = None
            elif line.startswith('id:'):
                current_term = line.split('id:')[1].strip()
                if current_term not in parent_child_map:
                    parent_child_map[current_term] = []
            elif line.startswith('is_a:'):
                parent = line.split('is_a:')[1].split('!')[0].strip()
                if parent not in parent_child_map:
                    parent_child_map[parent] = []
                if current_term:
                    parent_child_map[parent].append(current_term)
    return parent_child_map

# ==========================================
# KROK 1: Wczytanie i podział danych
# ==========================================
print("1. Wczytywanie danych...")
train_full_df = pd.read_parquet('/content/Hackathon_Task_1/Hackathon_task_1/data/chebi_dataset_train.parquet')

# Wyodrębnienie kolumn
smiles_full = train_full_df['SMILES'].tolist()
y_full = train_full_df.drop(columns=['SMILES', 'mol_id']).values
class_names = train_full_df.drop(columns=['SMILES', 'mol_id']).columns.tolist()

print("   -> Dzielenie na zbiór treningowy i testowy (80/20)...")
smiles_train, smiles_test, y_train, y_test = train_test_split(
    smiles_full, y_full, test_size=0.1, random_state=42
)

1. Wczytywanie danych...
   -> Dzielenie na zbiór treningowy i testowy (80/20)...


In [18]:

# # # ==========================================
# # # KROK 2: Ekstrakcja cech (scikit-fingerprints)
# # # ==========================================
# print("2. Generowanie wektorów MACCS Fingerprint...")
# maccs = MACCSFingerprint()

# X_train = maccs.transform(smiles_train)
# X_test = maccs.transform(smiles_test)

In [19]:
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, rdMolDescriptors
from skfp.fingerprints import MACCSFingerprint, ECFPFingerprint
from joblib import Parallel, delayed

# Inicjalizacja z n_jobs=-1 dla pełnej mocy procesora
maccs_gen = MACCSFingerprint(n_jobs=-1)
# Zostawiam Morgana (ECFP), bo genialnie łapie strukturę
morgan_gen = ECFPFingerprint(radius=2, fp_size=2048, n_jobs=-1)

# Twój zoptymalizowany zestaw deskryptorów
def get_advanced_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros(10)

    try:
        # 7 PhysChem Descriptors
        wt = Descriptors.MolWt(mol) / 1000            # Normalized Weight
        logp = Descriptors.MolLogP(mol) / 10          # Hydrophobicity
        hbd = Lipinski.NumHDonors(mol) / 10           # H-Bond Donors
        hba = Lipinski.NumHAcceptors(mol) / 10        # H-Bond Acceptors
        tpsa = Descriptors.TPSA(mol) / 200            # Polar Surface Area
        fsp3 = Lipinski.FractionCSP3(mol)             # Complexity (sp3 fraction)

        # 4 Topological/Ring Descriptors
        rings = Lipinski.RingCount(mol) / 10          # Total Rings
        arom_rings = Lipinski.NumAromaticRings(mol) / 10
        hetero_atoms = Lipinski.NumHeteroatoms(mol) / 20
        bridgeheads = rdMolDescriptors.CalcNumBridgeheadAtoms(mol) / 5

        return [wt, logp, hbd, hba, tpsa, fsp3, rings, arom_rings, hetero_atoms, bridgeheads]
    except:
        return np.zeros(10)

def extract_all_features(smiles_list):
    print(f"Przetwarzanie {len(smiles_list)} cząsteczek...")

    print("Obliczanie kluczy MACCS...")
    X_maccs = maccs_gen.transform(smiles_list)

    print("Obliczanie fingerprintów Morgana (ECFP)...")
    X_morgan = morgan_gen.transform(smiles_list)

    print("Obliczanie Twoich zaawansowanych deskryptorów na wszystkich rdzeniach...")
    X_desc_list = Parallel(n_jobs=-1)(delayed(get_advanced_descriptors)(s) for s in smiles_list)
    X_desc = np.array(X_desc_list)

    print("Łączenie wszystkich cech w jedną macierz...")
    X = np.concatenate([X_maccs, X_morgan, X_desc], axis=1)

    # Ostatnie zabezpieczenie przed błędem z PyTorcha
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    print(f"Gotowe! Ostateczny kształt macierzy: {X.shape}")
    return X

# --- URUCHOMIENIE ---
X_train = extract_all_features(smiles_train)
X_test = extract_all_features(smiles_test)

Przetwarzanie 30301 cząsteczek...
Obliczanie kluczy MACCS...
Obliczanie fingerprintów Morgana (ECFP)...
Obliczanie Twoich zaawansowanych deskryptorów na wszystkich rdzeniach...
Łączenie wszystkich cech w jedną macierz...
Gotowe! Ostateczny kształt macierzy: (30301, 2224)
Przetwarzanie 3367 cząsteczek...
Obliczanie kluczy MACCS...
Obliczanie fingerprintów Morgana (ECFP)...
Obliczanie Twoich zaawansowanych deskryptorów na wszystkich rdzeniach...
Łączenie wszystkich cech w jedną macierz...
Gotowe! Ostateczny kształt macierzy: (3367, 2224)


In [20]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import pandas as pd

# ==========================================
# KROK 3: Trening modelu (Sieć Neuronowa / PyTorch GPU)
# ==========================================
print("3. Trenowanie modelu (PyTorch GPU)...")

# Wykrywanie i ustawianie karty graficznej (GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"   -> Używane urządzenie: {device}")

valid_cols_mask = np.array([len(np.unique(y_train[:, i])) > 1 for i in range(y_train.shape[1])])
constant_classes = {class_names[i]: y_train[0, i] for i in range(len(class_names)) if not valid_cols_mask[i]}
print(f"   -> Pomijanie {len(constant_classes)} stałych klas w zbiorze treningowym...")

y_train_filtered = y_train[:, valid_cols_mask]
valid_class_names = [class_names[i] for i in range(len(class_names)) if valid_cols_mask[i]]

# Konwersja danych wejściowych na Tensory (format wymagany przez PyTorch)
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train_filtered, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)

# Przygotowanie paczek danych (Dataloaders)
batch_size = 256
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_t), batch_size=batch_size, shuffle=False)

# Definicja sieci (odpowiednik (512, 256) z MLPClassifier) - POPRAWIONA
model = nn.Sequential(
    nn.Linear(X_train_t.shape[1], 1024),
    nn.ReLU(),
    nn.Linear(1024, 512), # Usunięto nadmiarowe 512
    nn.ReLU(),
    nn.Linear(512, y_train_t.shape[1])
).to(device)

# Funkcja straty dostosowana do klasyfikacji wieloetykietowej (multi-label)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)

# Pętla treningowa
epochs = 200
model.train()

for epoch in range(epochs):
    epoch_loss = 0.0 # Zmienna do kumulowania straty w danej epoce

    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        # Wyciągamy wartość liczbową straty z tensora i dodajemy do sumy
        epoch_loss += loss.item()

    # Obliczamy średnią stratę dla całej epoki
    avg_loss = epoch_loss / len(train_loader)

    # Wyświetlamy wynik na ekranie
    print(f"Epoka [{epoch+1}/{epochs}] | Strata (Trening): {avg_loss:.4f}")

print("4. Generowanie predykcji...")
model.eval()
preds_list = []

with torch.no_grad():
    for inputs in test_loader:
        # DataLoader domyślnie zwraca tu krotkę z jednym elementem, więc bierzemy [0]
        inputs = inputs[0].to(device)
        outputs = model(inputs)
        # Przekształcamy wyniki na prawdopodobieństwa (0-1) za pomocą funkcji Sigmoid
        probs = torch.sigmoid(outputs)
        preds_list.append(probs.cpu().numpy())

# Połączenie paczek wyników w jedną macierz
proba_preds_filtered = np.vstack(preds_list)

# Odtworzenie pełnej ramki danych tak jak poprzednio
preds_df = pd.DataFrame(proba_preds_filtered, columns=valid_class_names)
for cls_name, const_val in constant_classes.items():
    preds_df[cls_name] = float(const_val)

# Upewniamy się, że kolumny są w tej samej kolejności co oryginalne y_test
preds_df = preds_df[class_names]

3. Trenowanie modelu (PyTorch GPU)...
   -> Używane urządzenie: cuda
   -> Pomijanie 1 stałych klas w zbiorze treningowym...
Epoka [1/200] | Strata (Trening): 0.1690
Epoka [2/200] | Strata (Trening): 0.0688
Epoka [3/200] | Strata (Trening): 0.0497
Epoka [4/200] | Strata (Trening): 0.0412
Epoka [5/200] | Strata (Trening): 0.0359
Epoka [6/200] | Strata (Trening): 0.0320
Epoka [7/200] | Strata (Trening): 0.0289
Epoka [8/200] | Strata (Trening): 0.0263
Epoka [9/200] | Strata (Trening): 0.0241
Epoka [10/200] | Strata (Trening): 0.0222
Epoka [11/200] | Strata (Trening): 0.0204
Epoka [12/200] | Strata (Trening): 0.0189
Epoka [13/200] | Strata (Trening): 0.0174
Epoka [14/200] | Strata (Trening): 0.0161
Epoka [15/200] | Strata (Trening): 0.0149
Epoka [16/200] | Strata (Trening): 0.0139
Epoka [17/200] | Strata (Trening): 0.0130
Epoka [18/200] | Strata (Trening): 0.0122
Epoka [19/200] | Strata (Trening): 0.0113
Epoka [20/200] | Strata (Trening): 0.0106
Epoka [21/200] | Strata (Trening): 0.0100
Ep

In [21]:

# ==========================================
# KROK 4: Post-processing (Logika DAG)
# ==========================================
print("5. Wgrywanie hierarchii i naprawa niekonsekwencji...")
parent_child_map = parse_obo_hierarchy('/content/Hackathon_Task_1/Hackathon_task_1/data/chebi_classes.obo')

corrections_made = 0
for parent, children in parent_child_map.items():
    if parent in class_names:
        for child in children:
            if child in class_names:
                mask = preds_df[child] > preds_df[parent]
                if mask.any():
                    preds_df.loc[mask, child] = preds_df.loc[mask, parent]
                    corrections_made += mask.sum()

print(f"   -> Wprowadzono {corrections_made} poprawek logicznych.")

5. Wgrywanie hierarchii i naprawa niekonsekwencji...
   -> Wprowadzono 475800 poprawek logicznych.


In [22]:
# ==========================================
# KROK 5: Binarizacja i ewaluacja (F1 Macro)
# ==========================================
print("6. Binarne formatowanie i obliczanie F1 Macro...")
final_preds_binary = (preds_df >= 0.5).astype(int)

# Konwersja do numpy array w celu obliczenia błędu
y_pred_array = final_preds_binary.values

# Obliczenie Macro-averaged F1 score
f1_macro = f1_score(y_test, y_pred_array, average='macro')

print("="*40)
print(f" WYNIK WALIDACJI (Macro F1 Score): {f1_macro:.4f}")
print("="*40)

6. Binarne formatowanie i obliczanie F1 Macro...
 WYNIK WALIDACJI (Macro F1 Score): 0.8158


array([[1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       ...,
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0]])

In [26]:
y_test_df = pd.DataFrame(y_test, columns=class_names)
y_test_df

,class_0,class_1,class_2,class_3,class_4,class_5,class_6,class_7,class_8,class_9,...,class_490,class_491,class_492,class_493,class_494,class_495,class_496,class_497,class_498,class_499
0,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,1,1,1,1,1,1,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
2,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
3,1,1,1,1,1,1,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
4,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3362,1,1,1,1,1,1,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
3363,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
3364,1,1,1,1,0,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
3365,1,1,1,1,1,1,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
y_test_df.to_parquet("", index=False)